# Repository inventory

A live manifest of what this repository tracks: the data files under `data/`
and the exported figures under `figures/`, both listed straight from disk so
the page always matches what is committed. Filenames carry processing
date-stamps that change on reprocessing, so this page is generated, not
hand-maintained. Re-run it after regenerating data or figures to refresh it.

The raw `00-*` inputs are tracked for provenance and never written to; the
derived stages (`01-` onward) are regenerated by the
[pipeline notebooks](../docs/notebooks.md) and tracked as a committed
snapshot. See [Data](../docs/data.md) for what each stage means and how it
maps onto the [processing versions](../docs/processing-versions.md).

## Imports and helpers

In [ ]:
import base64
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown, HTML

NB_START = datetime.now()  # notebook start time (reported in the last cell)

# Tracked datasets live under data/ (raw 00-* inputs and derived stages);
# exported figures live under figures/.
DATA = Path("../data")
FIGDIR = Path("../figures")


def human_size(n):
    """Human-readable byte size."""
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024


def scenario_of(name):
    """Leading scenario code in a filename (e.g. 'QCL-1'), or '' if none."""
    m = re.match(r"([A-Z]+-\d+)", name)
    return m.group(1) if m else ""


def folder_table(folder, pattern="*"):
    """One row per file in `folder` matching `pattern`: name, scenario, size."""
    rows = [
        {"file": f.name, "scenario": scenario_of(f.name),
         "size": human_size(f.stat().st_size)}
        for f in sorted(Path(folder).glob(pattern)) if f.is_file()
    ]
    return pd.DataFrame(rows)

## EddyPro flux output (`00-eddypro_fluxes_level-1/`)

One FLUXNET-format CSV per scenario (`*-1` to `*-4`); the PWB scenario
(`*-5`) has no flux output yet. These are the raw inputs to
[notebook 01](01_read_fluxes_to_parquet.ipynb).

In [ ]:
folder_table(DATA / "00-eddypro_fluxes_level-1", "*_adv.csv")

## EddyPro settings (`00-eddypro_settings/`)

The EddyPro project (`.eddypro`) and metadata (`.metadata`) files that
produced each flux output. They are the auditable definition of each
scenario's time-lag strategy; keep them in sync with the
[processing versions](../docs/processing-versions.md).

In [ ]:
# Map each scenario to its EddyPro project + metadata files and the lag
# strategy they encode. *-5 (PWB) has no EddyPro settings; it removes the lag
# with diive's detect-and-remove workflow instead.
STRATEGY = {
    "-1": "Covariance maximization, 0 to 10 s, no default (OPENLAG)",
    "-2": "Covariance maximization, 0 to 10 s, default fallback (DEFAULT-10s)",
    "-3": "Covariance maximization, narrow window, default fallback (DEFAULT-NARROW)",
    "-4": "Constant lag from the Flux Product (CH-CHA)",
}
sdir = DATA / "00-eddypro_settings"
rows = []
for cscode in sorted({scenario_of(f.name) for f in sdir.glob("*.eddypro")}):
    suffix = "-" + cscode.split("-", 1)[1]
    epro = next(iter(sorted(sdir.glob(f"{cscode}_*.eddypro"))), None)
    meta = next(iter(sorted(sdir.glob(f"{cscode}_*.metadata"))), None)
    rows.append({
        "scenario": cscode,
        "strategy": STRATEGY.get(suffix, ""),
        ".eddypro": epro.name if epro else "(missing)",
        ".metadata": meta.name if meta else "(missing)",
    })
pd.DataFrame(rows)

## PWB time-lag summaries (`00-pwb_tlag_summary/`)

The PWB detect-and-remove output for the `*-5` scenario: one
`*_detect_and_remove_tlag_summary.csv` per analyzer (with a `*_columns.md`
describing its columns). Time-lag results only, no fluxes yet.

In [ ]:
folder_table(DATA / "00-pwb_tlag_summary")

## Meteo (`00-meteo/`)

Supporting meteorological data. Empty for now.

In [ ]:
meteo = folder_table(DATA / "00-meteo")
display(meteo) if len(meteo) else display(Markdown("_Empty: meteo data not added yet._"))

## Derived stages (tracked Parquet)

Regenerated by the pipeline notebooks from the `00-*` inputs, and committed
as a snapshot of every stage.

In [ ]:
DERIVED = [
    "01-eddypro_fluxes_level-1_parquet",
    "01-pwb_tlag_summary_parquet",
    "02-eddypro_fluxes_level-1_parquet_subsets",
    "04-flux-product-2025.3_subset_2024",
    "05-merged_variants_fluxproduct",
]
for sub in DERIVED:
    display(Markdown(f"**`{sub}/`**"))
    display(folder_table(DATA / sub, "*.parquet"))

## Figures (`figures/`)

Every exported figure in `figures/`, shown in a grid (globbed at run time, so
new figures appear automatically). Images are base64-embedded so the page is
self-contained in the built book. For the curated, captioned view see the
[Figure gallery](../docs/figure-gallery.md).

In [ ]:
EXTS = ("*.png", "*.jpg", "*.jpeg", "*.gif", "*.svg")
MIME = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
        ".gif": "image/gif", ".svg": "image/svg+xml"}

images = sorted(p for ext in EXTS for p in FIGDIR.glob(ext))
print(f"Found {len(images)} figure(s) in {FIGDIR}")

cards = []
for p in images:
    b64 = base64.b64encode(p.read_bytes()).decode()
    mime = MIME.get(p.suffix.lower(), "image/png")
    cards.append(
        '<figure style="margin:0">'
        f'<img src="data:{mime};base64,{b64}" '
        'style="width:100%;height:auto;border:1px solid #ccc;border-radius:4px"/>'
        f'<figcaption style="font-size:0.8em;text-align:center;word-break:break-all">{p.name}</figcaption>'
        '</figure>'
    )

if cards:
    grid = (
        '<div style="display:grid;'
        'grid-template-columns:repeat(auto-fill,minmax(320px,1fr));gap:1rem">'
        + "".join(cards) + "</div>"
    )
    display(HTML(grid))
else:
    print("No figures found yet. Run the plotting notebooks first.")

## Runtime

In [ ]:
NB_END = datetime.now()
print(f"Start:    {NB_START:%Y-%m-%d %H:%M:%S}")
print(f"End:      {NB_END:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:  {NB_END - NB_START}")